# Reference servers

This notebook connects one chatbot to more than one MCP server. The servers are the research server, plus two reference servers: fetch and filesystem.

The runnable client is [`mcp_chatbot.py`](./mcp_chatbot.py). It reads [`server_config.json`](./server_config.json).

## Related notes

- [Direct tool calling](./mcp.ipynb)
- [MCP server](./mcp-server.ipynb)
- [MCP client](./mcp-client.ipynb)
- [Prompts and resources](./prompts-and-resources.ipynb)
- [Subprocess](./subprocess.ipynb)


## Servers in the config

The [reference server list](https://github.com/modelcontextprotocol/servers) collects official servers and links to community servers.

This config uses two of them, plus the local research server.

- [fetch](https://github.com/modelcontextprotocol/servers/tree/main/src/fetch) downloads a URL and returns the page as Markdown.
- [filesystem](https://github.com/modelcontextprotocol/servers/tree/main/src/filesystem) reads and writes files in a directory you name.

`npx` and `uvx` install those servers when the client starts them. You do not install them first.

The filesystem argument `.` means the current directory. Run the client from `MCP/` so the filesystem server uses this folder.


## Server configuration

[`server_config.json`](./server_config.json):

```json
{
  "mcpServers": {
    "filesystem": {
      "command": "npx",
      "args": ["-y", "@modelcontextprotocol/server-filesystem", "."]
    },
    "research": {
      "command": "uv",
      "args": [
        "run",
        "--with",
        "mcp>=1.2,<2",
        "--with",
        "arxiv",
        "research_server.py"
      ]
    },
    "fetch": {
      "command": "uvx",
      "args": ["--with", "mcp<2", "mcp-server-fetch"]
    }
  }
}
```

`mcp` stays below version 2 for the research server and for fetch. Fetch installs its own copy, so the pin is on the `uvx` arguments. See the [28 July 2026 spec](https://blog.modelcontextprotocol.io/posts/2026-07-28/).


## What changed from the single-server client

[The MCP client](./mcp-client.ipynb) uses one `async with` block. That block closes its session on the way out. Several servers need a stack you can add to while the program is running.

1. `sessions` holds one `ClientSession` per server.
2. `available_tools` holds every tool from every connected server.
3. `tool_to_session` maps a tool name to the session that owns it.
4. `exit_stack` is an `AsyncExitStack`. It closes each client and session in reverse order.
5. `connect_to_servers` reads the config and calls `connect_to_server` once per entry.
6. `cleanup` closes the stack. Call it when the chat ends, including after an error.

Two servers must not expose the same tool name. This client keeps one session per name, so the later server would replace the earlier one.


## Client code

The cell matches [`mcp_chatbot.py`](./mcp_chatbot.py), except the file runs `main` as a script. Run the file from the terminal.


In [ ]:
"""Chatbot that connects to every server in server_config.json."""

import asyncio
import json
from contextlib import AsyncExitStack

from anthropic import Anthropic
from dotenv import load_dotenv
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

load_dotenv()


class MCP_ChatBot:
    def __init__(self):
        self.sessions: list[ClientSession] = []
        self.exit_stack = AsyncExitStack()
        self.anthropic = Anthropic()
        self.available_tools: list[dict] = []
        self.tool_to_session: dict[str, ClientSession] = {}

    async def connect_to_server(self, server_name: str, server_config: dict) -> None:
        """Connect to a single MCP server."""
        try:
            server_params = StdioServerParameters(**server_config)
            read, write = await self.exit_stack.enter_async_context(
                stdio_client(server_params)
            )
            session = await self.exit_stack.enter_async_context(
                ClientSession(read, write)
            )
            await session.initialize()
            self.sessions.append(session)

            response = await session.list_tools()
            print(
                f"\nConnected to {server_name} with tools:",
                [tool.name for tool in response.tools],
            )
            for tool in response.tools:
                self.tool_to_session[tool.name] = session
                self.available_tools.append(
                    {
                        "name": tool.name,
                        "description": tool.description,
                        "input_schema": tool.inputSchema,
                    }
                )
        except Exception as error:
            print(f"Failed to connect to {server_name}: {error}")

    async def connect_to_servers(self) -> None:
        """Connect to all configured MCP servers."""
        try:
            with open("server_config.json", "r") as file:
                data = json.load(file)
            for server_name, server_config in data.get("mcpServers", {}).items():
                await self.connect_to_server(server_name, server_config)
        except Exception as error:
            print(f"Error loading server configuration: {error}")
            raise

    async def process_query(self, query):
        messages = [{"role": "user", "content": query}]
        response = self.anthropic.messages.create(
            max_tokens=2024,
            model="claude-sonnet-4-6",
            tools=self.available_tools,
            messages=messages,
        )
        process_query = True
        while process_query:
            assistant_content = []
            for content in response.content:
                if content.type == "text":
                    print(content.text)
                    assistant_content.append(content)
                    if len(response.content) == 1:
                        process_query = False
                elif content.type == "tool_use":
                    assistant_content.append(content)
                    messages.append({"role": "assistant", "content": assistant_content})
                    print(f"Calling tool {content.name} with args {content.input}")
                    session = self.tool_to_session[content.name]
                    result = await session.call_tool(
                        content.name, arguments=content.input
                    )
                    messages.append(
                        {
                            "role": "user",
                            "content": [
                                {
                                    "type": "tool_result",
                                    "tool_use_id": content.id,
                                    "content": result.content,
                                }
                            ],
                        }
                    )
                    response = self.anthropic.messages.create(
                        max_tokens=2024,
                        model="claude-sonnet-4-6",
                        tools=self.available_tools,
                        messages=messages,
                    )
                    if (
                        len(response.content) == 1
                        and response.content[0].type == "text"
                    ):
                        print(response.content[0].text)
                        process_query = False

    async def chat_loop(self):
        """Run an interactive chat loop."""
        print("\nMCP Chatbot Started!")
        print("Type your queries or 'quit' to exit.")
        while True:
            try:
                query = input("\nQuery: ").strip()
                if query.lower() == "quit":
                    break
                await self.process_query(query)
                print("\n")
            except Exception as error:
                print(f"\nError: {error}")

    async def cleanup(self):
        """Close every client and session in reverse order."""
        await self.exit_stack.aclose()


async def main():
    chatbot = MCP_ChatBot()
    try:
        await chatbot.connect_to_servers()
        await chatbot.chat_loop()
    finally:
        await chatbot.cleanup()


## Run it

From `MCP/`, with `ANTHROPIC_API_KEY` in an untracked `.env` file:

```bash
uv run --with "mcp>=1.2,<2" --with arxiv --with anthropic --with python-dotenv mcp_chatbot.py
```

Type `quit` to exit.

Example prompts:

- Fetch `https://modelcontextprotocol.io/docs/concepts/architecture` and save a short summary as `mcp_summary.md`.
- Search for two papers on a topic you name. Write the findings to `results.txt`.

Model text can differ from run to run.

Worth a later edit: connect to the servers concurrently, and decide what to do when two servers share a tool name.
